# Agentic Debugging QLoRA Patch Pilot v1

This notebook prepares the frozen CommitPackFT corpus and executes **non-held-out smoke only**. The full 7B training run and the five held-out generations remain disabled until FirstMate approves the freeze record and smoke evidence.

In [ ]:
# Cell 1 — Install frozen user-space dependencies. Do not pin Colab's CUDA torch wheel.
%pip install -q \
  transformers==5.14.1 \
  datasets==5.0.0 \
  peft==0.20.0 \
  trl==1.8.0 \
  bitsandbytes==0.49.2 \
  accelerate==1.14.0 \
  huggingface_hub \
  safetensors

In [ ]:
# Cell 2 — Mount persistent external storage and identify the uploaded repository.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
REPOSITORY_ROOT = Path('/content/agentic-debugging-internship')  # upload or clone the frozen baseline here
DRIVE_ROOT = Path('/content/drive/MyDrive/agentic-debugging/qlora_patch_pilot_v1')
CORPUS_ROOT = DRIVE_ROOT / 'corpus'
SMOKE_ROOT = DRIVE_ROOT / 'smoke'
MODEL_CACHE = DRIVE_ROOT / 'model-cache'
for path in (CORPUS_ROOT, SMOKE_ROOT, MODEL_CACHE):
    path.mkdir(parents=True, exist_ok=True)
assert (REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/freeze_record.json').is_file()
%cd {REPOSITORY_ROOT}

In [ ]:
# Cell 3 — Verify all frozen local identities before data or model work.
import json, subprocess, sys
FREEZE = REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/freeze_record.json'
result = subprocess.run([
    sys.executable, 'scripts/qlora_patch_pilot.py', 'verify-freeze',
    '--repository-root', str(REPOSITORY_ROOT), '--freeze-record', str(FREEZE),
], check=True, text=True, capture_output=True)
print(result.stdout)
verification = json.loads(result.stdout)
freeze = json.loads(FREEZE.read_text(encoding='utf-8'))
assert freeze['repository_baseline']['base_commit'] == '66fb5d5'
assert freeze['repository_baseline']['relationship'] == 'required_ancestor'
assert freeze['scientific_gate']['final_training_authorized'] is False
assert freeze['scientific_gate']['held_out_generation_authorized'] is False
print(json.dumps(verification['runtime'], indent=2))

In [ ]:
# Cell 4 — Download only the pinned Python source file to Google Drive.
from huggingface_hub import hf_hub_download
raw_path = Path(hf_hub_download(
    repo_id=freeze['dataset']['repository'],
    repo_type='dataset',
    revision=freeze['dataset']['revision'],
    filename='data/python/data.jsonl',
    cache_dir=str(DRIVE_ROOT / 'hub-cache'),
))
print({'raw_path': str(raw_path), 'size_bytes': raw_path.stat().st_size})

In [ ]:
# Cell 5 — Deterministic transform, repository-grouped split, and leakage checks.
command = [
    sys.executable, 'scripts/qlora_patch_pilot.py', 'build-corpus',
    '--repository-root', str(REPOSITORY_ROOT),
    '--input-jsonl', str(raw_path),
    '--output-dir', str(CORPUS_ROOT),
    '--freeze-record', str(FREEZE),
    '--transformation-config', str(REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/transformation_config.json'),
    '--prompt-contract', str(REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/prompt_contract.json'),
]
result = subprocess.run(command, check=True, text=True, capture_output=True)
print(result.stdout)
print((CORPUS_ROOT / 'rejection_summary.json').read_text(encoding='utf-8'))
print((CORPUS_ROOT / 'dedup_report.json').read_text(encoding='utf-8'))

## Mandatory manual audit

Review the 50 records in `accepted_audit_sample.jsonl` against `accepted_audit.csv`, and the 25 records in `rejected_audit_sample.jsonl` against `rejected_audit.csv`. Fill `manual_verdict`, `manual_reason`, `reviewer`, and `reviewed_at`. Accepted rows must be confirmed with the exact verdict `ACCEPT`; rejected rows with the exact verdict `REJECT`; any other value, missing field, or contradictory combination fails the gate. Do not change the deterministic sample selection.

In [ ]:
# Cell 6 — Fail closed until 50 accepted and 25 rejected reviews are completed.
audit_command = [
    sys.executable, 'scripts/qlora_patch_pilot.py', 'validate-audits',
    '--output-dir', str(CORPUS_ROOT),
    '--transformation-config', str(REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/transformation_config.json'),
]
result = subprocess.run(audit_command, check=True, text=True, capture_output=True)
print(result.stdout)

In [ ]:
# Cell 7 — Record runtime identity and require a CUDA accelerator.
import importlib.metadata as md, platform, torch
runtime = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_runtime': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_total_memory_bytes': torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
    'packages': {name: md.version(name) for name in ['transformers','datasets','peft','trl','bitsandbytes','accelerate','huggingface_hub','safetensors']},
    'repository_verification': verification['runtime'],
}
(SMOKE_ROOT / 'runtime_environment.json').write_text(json.dumps(runtime, indent=2, sort_keys=True) + '\n')
print(json.dumps(runtime, indent=2))
assert torch.cuda.is_available(), 'A CUDA Colab runtime is required for the 7B QLoRA smoke.' 

In [ ]:
# Cell 8 — Load one non-held-out training example and prepare completion-only labels.
from datasets import load_dataset
from transformers import AutoTokenizer
training_cfg = json.loads((REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/training_config.json').read_text())
model_id = training_cfg['model_repository']
model_revision = training_cfg['model_revision']
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=model_revision, cache_dir=str(MODEL_CACHE), use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
raw_train = load_dataset('json', data_files=str(CORPUS_ROOT / 'train.jsonl'), split='train')
smoke_example = raw_train.select([0])

def tokenize_completion_only(example):
    prompt_text = tokenizer.apply_chat_template(example['prompt'], tokenize=False, add_generation_prompt=True)
    completion = example['completion']
    full_text = prompt_text + completion + (tokenizer.eos_token or '')
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
    encoded = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=training_cfg['sft']['max_length'])
    labels = [-100] * min(len(prompt_ids), len(encoded['input_ids'])) + encoded['input_ids'][len(prompt_ids):]
    encoded['labels'] = labels
    return encoded

tokenized_smoke = smoke_example.map(tokenize_completion_only, remove_columns=smoke_example.column_names)
assert any(label != -100 for label in tokenized_smoke[0]['labels'])
print({'input_tokens': len(tokenized_smoke[0]['input_ids']), 'completion_tokens': sum(x != -100 for x in tokenized_smoke[0]['labels'])})

In [ ]:
# Cell 9 — Load the pinned 7B checkpoint in 4-bit and attach LoRA.
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant = training_cfg['quantization']
quant_config = BitsAndBytesConfig(
    load_in_4bit=quant['load_in_4bit'],
    bnb_4bit_quant_type=quant['quant_type'],
    bnb_4bit_use_double_quant=quant['double_quant'],
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=model_revision, quantization_config=quant_config,
    device_map={'': 0}, cache_dir=str(MODEL_CACHE), dtype=compute_dtype,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_cfg = training_cfg['lora']
lora = LoraConfig(
    r=lora_cfg['r'], lora_alpha=lora_cfg['alpha'], lora_dropout=lora_cfg['dropout'],
    target_modules=lora_cfg['target_modules'], bias=lora_cfg['bias'], task_type=lora_cfg['task_type'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# Cell 10 — One-example, one-step real weight-update smoke. This is not the final experiment.
from dataclasses import dataclass
from transformers import Trainer, TrainingArguments
from agentic_debugger.training.patch_pilot import aggregate_lora_delta, snapshot_trainable_lora_parameters

@dataclass
class CompletionOnlyCollator:
    pad_token_id: int
    def __call__(self, features):
        max_len = max(len(item['input_ids']) for item in features)
        batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
        for item in features:
            pad = max_len - len(item['input_ids'])
            batch['input_ids'].append(item['input_ids'] + [self.pad_token_id] * pad)
            batch['attention_mask'].append(item['attention_mask'] + [0] * pad)
            batch['labels'].append(item['labels'] + [-100] * pad)
        return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}

lora_before = snapshot_trainable_lora_parameters(model)
assert lora_before, 'No trainable LoRA tensor was found before the smoke step.'
smoke_adapter = SMOKE_ROOT / 'adapter-one-step'
args = TrainingArguments(
    output_dir=str(SMOKE_ROOT / 'trainer-output'),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    max_steps=1,
    learning_rate=training_cfg['sft']['learning_rate'],
    optim=training_cfg['sft']['optim'],
    lr_scheduler_type=training_cfg['sft']['lr_scheduler_type'],
    warmup_ratio=0.0,
    logging_steps=1,
    save_strategy='no',
    report_to='none',
    fp16=compute_dtype == torch.float16,
    bf16=compute_dtype == torch.bfloat16,
    seed=training_cfg['sft']['seed'],
    data_seed=training_cfg['sft']['data_seed'],
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized_smoke, data_collator=CompletionOnlyCollator(tokenizer.pad_token_id))
train_result = trainer.train()
lora_delta = aggregate_lora_delta(lora_before, model)
assert lora_delta['trainable_tensors_checked'] == len(lora_before), 'Trainable LoRA tensor count changed during the step.'
assert lora_delta['changed_tensors'] > 0, 'No LoRA tensor changed during the smoke step.'
assert lora_delta['aggregate_delta_l2'] > 0.0, 'Aggregate LoRA delta is not positive.'
assert lora_delta['delta_finite'] is True, 'Aggregate LoRA delta is not finite.'
model.save_pretrained(smoke_adapter, safe_serialization=True)
tokenizer.save_pretrained(smoke_adapter)
smoke_train_record = {'train_loss': train_result.training_loss, **lora_delta, 'max_steps': 1}
(SMOKE_ROOT / 'one_step_training.json').write_text(json.dumps(smoke_train_record, indent=2, sort_keys=True) + '\n')
print(smoke_train_record)

In [ ]:
# Cell 11 — Reload the saved adapter and run deterministic non-held-out inference.
import gc
from peft import PeftModel
from agentic_debugger.training.patch_pilot import parse_unified_diff_strict, sha256_bytes

del trainer, model
gc.collect(); torch.cuda.empty_cache()
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=model_revision, quantization_config=quant_config,
    device_map={'': 0}, cache_dir=str(MODEL_CACHE), dtype=compute_dtype,
)
reloaded = PeftModel.from_pretrained(base_model, smoke_adapter, is_trainable=False)
reloaded.eval()
validation_rows = load_dataset('json', data_files=str(CORPUS_ROOT / 'validation.jsonl'), split='train')
probe = validation_rows[0]
prompt_text = tokenizer.apply_chat_template(probe['prompt'], tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors='pt').to('cuda')
with torch.inference_mode():
    generated = reloaded.generate(**inputs, do_sample=False, num_beams=1, max_new_tokens=512, use_cache=True)
raw_output = tokenizer.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
(SMOKE_ROOT / 'non_held_out_inference.txt').write_text(raw_output, encoding='utf-8')
try:
    parse_unified_diff_strict(raw_output, [probe['provenance'].get('file_path', probe['example_id'].rsplit(':', 1)[-1])])
    strict_parse_valid = True
    strict_parse_error = None
except Exception as exc:
    strict_parse_valid = False
    strict_parse_error = f'{type(exc).__name__}: {exc}'
record = {'strict_parse_valid': strict_parse_valid, 'strict_parse_error': strict_parse_error, 'output_sha256': sha256_bytes(raw_output.encode()), 'held_out_task_used': False}
(SMOKE_ROOT / 'inference_record.json').write_text(json.dumps(record, indent=2, sort_keys=True) + '\n')
print(record)

In [ ]:
# Cell 12 — Independently smoke the strict parser and the existing verifier on a synthetic non-held-out task.
known_patch = '--- a/counter.py\n+++ b/counter.py\n@@ -1,2 +1,2 @@\n def next_value(value: int) -> int:\n-    return value - 1\n+    return value + 1\n'
assert parse_unified_diff_strict(known_patch, ['counter.py']) == known_patch
verifier_result = subprocess.run([
    sys.executable, 'scripts/qlora_patch_pilot.py', 'verifier-smoke',
    '--repository-root', str(REPOSITORY_ROOT), '--output', str(SMOKE_ROOT / 'verifier_smoke.json'),
], check=True, text=True, capture_output=True)
print(verifier_result.stdout)

In [ ]:
# Cell 13 — Record checksums for all external smoke artifacts.
from agentic_debugger.training.patch_pilot import write_external_manifest
manifest = write_external_manifest(SMOKE_ROOT, configuration_identity=freeze['training']['sha256'], provenance_identity=f"{model_id}@{model_revision}", artifact_kind_prefix='smoke')
print(json.dumps(manifest, indent=2)[:4000])

## STOP — FirstMate review gate

Do not change either authorization flag in this notebook merely to continue. After FirstMate approves the freeze and smoke evidence, create a separately recorded authorized run configuration. The final run must consume the frozen train/validation files and generate each held-out output exactly once.

In [ ]:
# Cell 14 — Hard gate. Expected result in this phase: both assertions pass because execution remains disabled.
FINAL_TRAINING_AUTHORIZED = False
HELD_OUT_GENERATION_AUTHORIZED = False
assert FINAL_TRAINING_AUTHORIZED is freeze['scientific_gate']['final_training_authorized'] is False
assert HELD_OUT_GENERATION_AUTHORIZED is freeze['scientific_gate']['held_out_generation_authorized'] is False
print('STOPPED_BEFORE_FINAL_TRAINING_AND_HELD_OUT_GENERATION')

## Deferred authorized cells

The final-training and held-out-evaluation implementation is intentionally not executed in this phase. It will:

1. load the frozen 1,500/200 corpus, or the accepted 1,000/150 minimum without weakening filters;
2. train one epoch from the pinned 7B base checkpoint;
3. select the final adapter without any held-out outcome;
4. save configuration, logs, adapter, sizes, and checksums externally;
5. generate one base and one tuned candidate per frozen task using the identical prompt and generation contract;
6. save raw outputs before parsing;
7. submit the saved patches to the existing verifier;
8. allow verifier-only reruns, but refuse model regeneration after an outcome record exists.